# Module 5.2 — Similarity Score Threshold Retrieval

Instead of always returning k documents, return only those **above a relevance threshold**.
This prevents injecting irrelevant context into the LLM prompt.

| Method | Returns | Use when |
|---|---|---|
| `similarity_search(k=5)` | Always exactly k docs | You always need k results |
| `similarity_search_with_relevance_scores` | Docs with scores | You need to inspect scores |
| `as_retriever(search_type='similarity_score_threshold')` | Docs above threshold | Quality > quantity |

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

docs = [
    Document(page_content='The Eiffel Tower is located in Paris, France.'),
    Document(page_content='The Colosseum is an ancient amphitheater in Rome, Italy.'),
    Document(page_content='Mount Everest is the highest mountain on Earth, located in Nepal.'),
    Document(page_content='The Amazon River is the largest river by discharge in the world.'),
    Document(page_content='Python was created by Guido van Rossum in 1991.'),
]
vs = Chroma.from_documents(docs, embeddings, collection_name='threshold_demo')

query = 'Famous European landmarks'

# ── Standard k=5 (always returns 5) ─────────────────────────────────────────
std = vs.similarity_search(query, k=5)
print(f'Standard (k=5): {len(std)} results')
for d in std:
    print(f'  • {d.page_content}')

# ── Scores inspection ─────────────────────────────────────────────────────────
print('\nWith relevance scores:')
scored = vs.similarity_search_with_relevance_scores(query, k=5)
for doc, score in scored:
    flag = '✅' if score >= 0.75 else '❌'
    print(f'  {flag} [{score:.4f}] {doc.page_content[:60]}')

# ── Threshold retriever (only returns docs above 0.75) ───────────────────────
threshold_retriever = vs.as_retriever(
    search_type='similarity_score_threshold',
    search_kwargs={'score_threshold': 0.75, 'k': 5}
)
thresh_results = threshold_retriever.invoke(query)
print(f'\nThreshold retriever (≥0.75): {len(thresh_results)} results')
for d in thresh_results:
    print(f'  • {d.page_content}')

In [ ]:
# ── Threshold sweep: find the right value ─────────────────────────────────────
print(f'Threshold sweep for query: "{query}"\n')
print(f'  {"Threshold":>12} | {"Docs returned":>14}')
print('  ' + '-'*30)
for threshold in [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]:
    r = vs.as_retriever(
        search_type='similarity_score_threshold',
        search_kwargs={'score_threshold': threshold, 'k': 10}
    ).invoke(query)
    print(f'  {threshold:>12.2f} | {len(r):>14}')